# Training_File_Sweep.ipynb — STEP 2B parametric sweep

**Scopo**: orchestrare N esperimenti consecutivi con configurazioni diverse di capacity
(`cf_hidden_size`, `cf_rank`) e scenario (`scenario_mix`, `cut_in_ratio`).

**Differenze rispetto a `Training_File.ipynb`**:
- `Training_File.ipynb` → 1 esperimento singolo, analisi visiva approfondita
- `Training_File_Sweep.ipynb` → N esperimenti in sequenza, summary aggregato

**Strategie chiave**:
1. **Cache condivisa per scenario**: runs con stesso `(n_train, scenario_mix, cut_in_ratio)` riusano la stessa cache → 1 generazione vs N. Capacity sweep su highway-only riusa la cache 5 volte.
2. **Push per-run** (non a fine sweep): se Azure crasha a metà, i results parziali sono già su GitHub.
3. **Continue-on-failure**: se un run fallisce (preflight o training), si logga il problema e si passa al successivo.
4. **Sanity check pre-sweep**: verifica che lo stesso codice non sia già stato runnato (TAG collision check).

In [ ]:
# ===========================================================
# CELLA 0 — Bootstrap (sync remote + sanity check imports)
# ===========================================================
import os, sys, glob, json, subprocess, datetime, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML, Image

# Sync con remote (essenziale per avere ultimi fix)
print('🔄 git pull --no-rebase --no-edit origin main...')
!git pull --no-rebase --no-edit origin main

print('\n📍 Commit attuale:')
!git log --oneline -3

# Sanity check infrastruttura
print('\n🔧 Sanity check:')
!python -c "from train import BatchCSVLogger; from core.network import CF_FSNN_Net; \
            m = CF_FSNN_Net(hidden_size=64, rank=16); \
            assert m.hidden_size == 64 and m.rank == 16, 'sweep API non funzionante'; \
            print('✅ CF_FSNN_Net accetta hidden_size/rank kwargs')"

assert os.path.isfile('scripts/preflight.py'), 'scripts/preflight.py NON trovato'
print('✅ scripts/preflight.py presente')

In [ ]:
# ===========================================================
# CELLA 1 — SWEEP CONFIG: lista esperimenti da eseguire
# ===========================================================
# Ogni elemento è un dict che OVERRIDA i default. Campi minimi richiesti: tag.
# Tutti gli altri usano i default qui sotto.
#
# Strategia STEP 2B:
#   Block A — capacity sweep su highway-only (effetto puro di capacity)
#   Block B — scenario diversity a capacity intermedia (effetto puro di scenario)
#
# Cache condivisione: runs con stesso (n_train, scenario_mix, cut_in_ratio) → stessa
# cache. Block A riusa la stessa cache 5 volte (highway, cut0).

# ── DEFAULTS condivisi (overridabili da ogni run) ──────────────
DEFAULTS = {
    # Training duration — fast iteration mode (STEP 2A validato: convergenza in ~17 min)
    'epochs':       10,
    'n_train':      500,
    'n_val':        100,
    # Scheduler
    'scheduler':    'onecycle',
    'max_lr':       2e-3,
    'T0':           5,
    'lr':           1e-3,
    # Sequenza
    'seq_len':      50,
    'batch_size':   64,
    # Pesi loss PINN
    'lambda_data':  1.0,
    'lambda_phys':  0.1,
    'lambda_ou':    0.05,
    'lambda_bc':    1.0,
    'optimizer':    'adam',
    # Capacity (default = config.py: hidden=32, rank=8)
    'cf_hidden_size': None,   # None → usa CF_HIDDEN_SIZE di config.py (= 32)
    'cf_rank':        None,   # None → usa CF_RANK di config.py        (= 8)
    # Scenario
    'scenario_mix':   'highway',
    'cut_in_ratio':   0.0,
    # Early stopping (aggressivo per fast iteration)
    'early_stop_patience': 2,
    'early_stop_delta':    0.005,
    # Diagnostica
    'max_inf_streak': 20,
}

# ── SWEEP_PLAN: lista degli esperimenti ───────────────────────
SWEEP_PLAN = [
    # ── BLOCK A: capacity sweep, highway-only (cache condivisa) ──
    {'tag': 'P9_S2B_h32_r8_hw',   'cf_hidden_size':  32, 'cf_rank':  8},  # baseline = P9_S2A
    {'tag': 'P9_S2B_h48_r12_hw',  'cf_hidden_size':  48, 'cf_rank': 12},
    {'tag': 'P9_S2B_h64_r16_hw',  'cf_hidden_size':  64, 'cf_rank': 16},
    {'tag': 'P9_S2B_h96_r24_hw',  'cf_hidden_size':  96, 'cf_rank': 24},
    {'tag': 'P9_S2B_h128_r32_hw', 'cf_hidden_size': 128, 'cf_rank': 32},

    # ── BLOCK B: scenario diversity a h64_r16 (presumibile sweet spot) ──
    # Nota: se Block A indicasse un sweet spot diverso, modificare h/r qui.
    {'tag': 'P9_S2B_h64_r16_urban',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'urban'},
    {'tag': 'P9_S2B_h64_r16_truck',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'truck'},
    {'tag': 'P9_S2B_h64_r16_mixed',   'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'mixed'},  # default 4-scenario
    {'tag': 'P9_S2B_h64_r16_hwcut15', 'cf_hidden_size': 64, 'cf_rank': 16, 'scenario_mix': 'highway', 'cut_in_ratio': 0.15},
]

# Controllo per-run
RUN_PREFLIGHT  = True    # doppio smoke prima del FULL per ogni run
PUSH_PER_RUN   = True    # push dei results di OGNI run (non solo a fine sweep)
SKIP_IF_EXISTS = True    # se results/<tag>/ esiste già, salta
STOP_ON_FAIL   = False   # False = continua, True = ferma sweep al primo fail

# Report sweep plan
print(f'📋 SWEEP PLAN: {len(SWEEP_PLAN)} esperimenti\n')
for i, run in enumerate(SWEEP_PLAN, 1):
    cfg = {**DEFAULTS, **run}
    print(f"  {i:2d}. {cfg['tag']:<32}  h={cfg['cf_hidden_size']}, r={cfg['cf_rank']}, "
          f"scen={cfg['scenario_mix']}, cut={cfg['cut_in_ratio']}")

print(f'\n⚙️  Flags: preflight={RUN_PREFLIGHT}, push_per_run={PUSH_PER_RUN}, '
      f'skip_if_exists={SKIP_IF_EXISTS}, stop_on_fail={STOP_ON_FAIL}')
print(f'⏱️  Tempo stimato Azure CPU: ~5h totali (ciascun run 17-75 min in base a capacity)')

In [ ]:
# ===========================================================
# CELLA 2 — Helper: cache path normalizer + run executor
# ===========================================================
def _safe_scenario(s):
    """Normalizza scenario_mix per usarlo in un nome file.
    'highway:0.7,urban:0.3' → 'highway_0.7_urban_0.3'
    """
    return s.replace(':', '_').replace(',', '_').replace(' ', '')

def _cache_path(cfg):
    """Path cache deterministico da (n_train, scenario, cut_in). Runs con stessa
    config dataset riusano la stessa cache anche con h/r diversi."""
    s = _safe_scenario(cfg['scenario_mix'])
    return f"data/cache_{cfg['n_train']}_{s}_cut{cfg['cut_in_ratio']}.pt"

def _build_cli_args(cfg, cache_path):
    """Costruisce gli argv di train.py da un cfg dict."""
    args = [
        '--epochs',         str(cfg['epochs']),
        '--scheduler',      cfg['scheduler'],
        '--seq_len',        str(cfg['seq_len']),
        '--batch_size',     str(cfg['batch_size']),
        '--lambda_data',    str(cfg['lambda_data']),
        '--lambda_phys',    str(cfg['lambda_phys']),
        '--lambda_ou',      str(cfg['lambda_ou']),
        '--lambda_bc',      str(cfg['lambda_bc']),
        '--optimizer',      cfg['optimizer'],
        '--max_inf_streak', str(cfg['max_inf_streak']),
        '--data_cache',     cache_path,
        '--tag',            cfg['tag'],
        '--n_train',        str(cfg['n_train']),
        '--n_val',          str(cfg['n_val']),
        '--scenario_mix',   cfg['scenario_mix'],
        '--early_stop_patience', str(cfg['early_stop_patience']),
        '--early_stop_delta',    str(cfg['early_stop_delta']),
    ]
    if cfg.get('cut_in_ratio') is not None:
        args += ['--cut_in_ratio', str(cfg['cut_in_ratio'])]
    if cfg.get('cf_hidden_size') is not None:
        args += ['--cf_hidden_size', str(cfg['cf_hidden_size'])]
    if cfg.get('cf_rank') is not None:
        args += ['--cf_rank', str(cfg['cf_rank'])]
    if cfg.get('lambda_sr') is not None:
        args += ['--lambda_sr', str(cfg['lambda_sr'])]
    # Scheduler-specific
    if cfg['scheduler'] == 'onecycle':
        args += ['--max_lr', str(cfg['max_lr'])]
    elif cfg['scheduler'] == 'cosine':
        args += ['--T0', str(cfg['T0'])]
    elif cfg['scheduler'] == 'plateau':
        args += ['--lr', str(cfg['lr'])]
    return args

def _push_results(cfg):
    """Copia checkpoints/<tag>/ → results/<tag>/, commit+pull-before-push."""
    tag     = cfg['tag']
    src_dir = f'checkpoints/{tag}'
    dst_dir = f'results/{tag}'
    if not os.path.isdir(src_dir):
        print(f'   ⚠️  {src_dir} mancante — niente da pushare')
        return False

    if os.path.isdir(dst_dir):
        shutil.rmtree(dst_dir)
    os.makedirs(f'{dst_dir}/plots', exist_ok=True)
    for f in glob.glob(f'{src_dir}/*.csv') + glob.glob(f'{src_dir}/*.json'):
        shutil.copy2(f, dst_dir)
    for f in glob.glob(f'{src_dir}/plots/*.png'):
        shutil.copy2(f, f'{dst_dir}/plots/')
    if os.path.isfile(f'{src_dir}/crash_model.pt'):
        import torch
        ck = torch.load(f'{src_dir}/crash_model.pt', map_location='cpu', weights_only=False)
        with open(f'{dst_dir}/CRASH_INFO.txt', 'w') as f:
            f.write(f"Crash\nEpoch: {ck.get('epoch','?')}\nval_loss: {ck.get('val_loss','?')}\n")

    # Commit + pull-before-push (autoresolve fast-forward su Azure)
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (f"results: {tag} ({ts})\n\n"
           f"Capacity: h={cfg.get('cf_hidden_size')}, r={cfg.get('cf_rank')}\n"
           f"Scenario: {cfg['scenario_mix']}, cut_in={cfg['cut_in_ratio']}\n"
           f"Sweep run via Training_File_Sweep.ipynb.\n")
    with open('/tmp/sweep_commit_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst_dir}
    !git commit -F /tmp/sweep_commit_msg.txt 2>&1 | tail -3
    !rm /tmp/sweep_commit_msg.txt
    print('   🔄 git pull --no-rebase --no-edit origin main')
    !git pull --no-rebase --no-edit origin main 2>&1 | tail -3
    print('   📤 git push origin main')
    !git push origin main 2>&1 | tail -3
    return True

print('✅ Helper functions definite')

In [ ]:
# ===========================================================
# CELLA 3 — ESECUZIONE SWEEP
# Per ogni run: preflight (opzionale) → train.py → push results (opzionale)
# Robust: continue-on-failure (default), tracking di sweep state in DataFrame.
# ===========================================================
sweep_results = []   # raccoglie risultati per il summary finale
sweep_start  = time.time()

for i, run_override in enumerate(SWEEP_PLAN, 1):
    cfg = {**DEFAULTS, **run_override}
    tag = cfg['tag']
    
    print('\n' + '=' * 78)
    print(f'🚀 RUN {i}/{len(SWEEP_PLAN)}: {tag}')
    print(f'   Capacity: hidden={cfg["cf_hidden_size"]}, rank={cfg["cf_rank"]}')
    print(f'   Scenario: {cfg["scenario_mix"]}, cut_in={cfg["cut_in_ratio"]}')
    print(f'   Cache:    {_cache_path(cfg)}')
    print('=' * 78)
    
    run_start = time.time()
    record = {
        'tag': tag,
        'cf_hidden_size': cfg['cf_hidden_size'],
        'cf_rank':        cfg['cf_rank'],
        'scenario_mix':   cfg['scenario_mix'],
        'cut_in_ratio':   cfg['cut_in_ratio'],
        'status':         'unknown',
        'elapsed_s':      0.0,
        'best_val':       np.nan,
        'best_epoch':     np.nan,
        'n_epochs':       0,
        'n_inf_batches':  0,
        'spike_pct':      np.nan,
    }
    
    # ── Skip se results/<tag>/ esiste già (riprese di sweep parziali) ────
    if SKIP_IF_EXISTS and os.path.isdir(f'results/{tag}'):
        print(f'   ⏭️  SKIP: results/{tag}/ già presente (set SKIP_IF_EXISTS=False per forzare)')
        record['status'] = 'skipped_existing'
        sweep_results.append(record)
        continue
    
    cache_path = _cache_path(cfg)
    cli_args   = _build_cli_args(cfg, cache_path)
    
    # ── PREFLIGHT (doppio smoke) ─────────────────────────────────────
    preflight_ok = True
    if RUN_PREFLIGHT:
        pf_extra = ['--seq_len', str(cfg['seq_len'])]
        if cfg['scheduler'] == 'onecycle':
            pf_extra += ['--max_lr', str(cfg['max_lr'])]
        if cfg['cf_hidden_size'] is not None:
            pf_extra += ['--cf_hidden_size', str(cfg['cf_hidden_size'])]
        if cfg['cf_rank'] is not None:
            pf_extra += ['--cf_rank', str(cfg['cf_rank'])]
        pf_cmd = ['python', 'scripts/preflight.py', '--base_tag', tag, '--extra'] + pf_extra
        print(f'\n🚦 Preflight:  {" ".join(pf_cmd)}')
        pf_res = subprocess.run(pf_cmd, capture_output=False)
        preflight_ok = (pf_res.returncode == 0)
        if not preflight_ok:
            print(f'   ❌ PREFLIGHT FAIL — skipping FULL train per questo run')
            record['status']    = 'preflight_fail'
            record['elapsed_s'] = time.time() - run_start
            sweep_results.append(record)
            if STOP_ON_FAIL:
                print('   🛑 STOP_ON_FAIL=True → sweep interrotto')
                break
            continue
    
    # ── FULL TRAIN ──────────────────────────────────────────────────
    full_cmd = ['python', 'train.py'] + cli_args
    print(f'\n▶️  Full train: {" ".join(full_cmd)}')
    full_res = subprocess.run(full_cmd, capture_output=False)
    full_ok  = (full_res.returncode == 0)
    
    # Estrai metriche anche se FULL FAIL (potrebbe avere salvato qualche epoca)
    epoch_csv = f'checkpoints/{tag}/training_log.csv'
    batch_csv = f'checkpoints/{tag}/training_batch_log.csv'
    if os.path.isfile(epoch_csv):
        edf = pd.read_csv(epoch_csv)
        if len(edf) > 0:
            record['best_val']   = float(edf['val_total'].min())
            record['best_epoch'] = int(edf['val_total'].idxmin()) + 1
            record['n_epochs']   = int(len(edf))
    if os.path.isfile(batch_csv):
        bdf = pd.read_csv(batch_csv)
        record['n_inf_batches'] = int(bdf['is_inf_grad'].sum())
        record['spike_pct']     = float(bdf['spike_rate'].mean() * 100)
    
    record['status']    = 'ok' if full_ok else 'full_fail'
    record['elapsed_s'] = time.time() - run_start
    
    # ── PUSH PER-RUN ────────────────────────────────────────────────
    if PUSH_PER_RUN:
        print(f'\n📤 Push results/{tag}/...')
        _push_results(cfg)
    
    # Formatta best_val pre-print per evitare f-string conditional invalido
    best_val_str = f'{record["best_val"]:.4f}' if not np.isnan(record['best_val']) else 'n/a'
    print(f'\n✅ Run {i}/{len(SWEEP_PLAN)} completato — status={record["status"]}, '
          f'best_val={best_val_str}, elapsed={record["elapsed_s"]/60:.1f}min')
    sweep_results.append(record)
    
    if not full_ok and STOP_ON_FAIL:
        print('   🛑 STOP_ON_FAIL=True → sweep interrotto')
        break

sweep_total_min = (time.time() - sweep_start) / 60
print('\n' + '=' * 78)
print(f'🏁 SWEEP COMPLETATO — {len(sweep_results)}/{len(SWEEP_PLAN)} runs in {sweep_total_min:.1f} min')
print('=' * 78)

In [ ]:
# ===========================================================
# CELLA 4 — SUMMARY: tabella riassuntiva sweep
# ===========================================================
if not sweep_results:
    print('❌ Nessun risultato — esegui Cella 3 prima.')
else:
    df = pd.DataFrame(sweep_results)
    
    # Formatting
    df_disp = df.copy()
    df_disp['elapsed_min'] = (df_disp['elapsed_s'] / 60).round(1)
    df_disp['best_val']    = df_disp['best_val'].apply(lambda x: f'{x:.4f}' if pd.notna(x) else '-')
    df_disp['spike_pct']   = df_disp['spike_pct'].apply(lambda x: f'{x:.2f}%' if pd.notna(x) else '-')
    df_disp['epochs']      = df_disp['n_epochs'].astype(str) + '/' + df_disp.apply(
        lambda r: '?', axis=1)
    
    cols = ['tag','status','cf_hidden_size','cf_rank','scenario_mix','cut_in_ratio',
            'best_val','best_epoch','n_epochs','n_inf_batches','spike_pct','elapsed_min']
    display(Markdown(f'### 📊 SWEEP SUMMARY (N={len(df)})'))
    display(df_disp[cols])
    
    # ── Best per category ────────────────────────────────────────
    ok_runs = df[df['status'] == 'ok']
    if len(ok_runs) > 0:
        display(Markdown('### 🏆 Highlights'))
        
        # Capacity sweep (highway-only)
        cap_runs = ok_runs[(ok_runs['scenario_mix'] == 'highway') & (ok_runs['cut_in_ratio'] == 0.0)]
        if len(cap_runs) > 0:
            best_cap = cap_runs.loc[cap_runs['best_val'].idxmin()]
            print(f"   🎯 Best capacity (highway):  {best_cap['tag']:<35} "
                  f"h={best_cap['cf_hidden_size']}, r={best_cap['cf_rank']}, "
                  f"val={best_cap['best_val']:.4f}")
            print(f"\n   📈 Capacity sweep (highway):")
            for _, r in cap_runs.sort_values('cf_hidden_size').iterrows():
                bar = '█' * int((0.30 - min(r['best_val'], 0.30)) * 200)
                print(f"     h={r['cf_hidden_size']:>3}, r={r['cf_rank']:>2}: "
                      f"val={r['best_val']:.4f} {bar}")
        
        # Scenario diversity
        scn_runs = ok_runs[~((ok_runs['scenario_mix'] == 'highway') & (ok_runs['cut_in_ratio'] == 0.0))]
        if len(scn_runs) > 0:
            print(f"\n   🌍 Scenario diversity:")
            for _, r in scn_runs.iterrows():
                print(f"     {r['scenario_mix']:<10} cut={r['cut_in_ratio']:.2f}: val={r['best_val']:.4f}")
    
    # ── Fail analysis ────────────────────────────────────────────
    fails = df[~df['status'].isin(['ok','skipped_existing'])]
    if len(fails) > 0:
        display(Markdown(f'### ⚠️  Fails: {len(fails)}'))
        display(fails[['tag','status','n_epochs','n_inf_batches','elapsed_min' if 'elapsed_min' in fails.columns else 'elapsed_s']])
    
    # Salva CSV per analisi offline
    sweep_csv = f'sweep_summary_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}.csv'
    df.to_csv(sweep_csv, index=False)
    print(f'\n💾 Summary salvato in: {sweep_csv}')